In [1]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [2]:
# import shutil
# from pathlib import Path
# from datetime import datetime

# # Paths
# HISTORY_PATH = Path("selection_history.json")
# TRAIN_DIR    = Path("training_sets")
# OUTPUT_DIR   = Path(".")

# # Backup folder with timestamp
# BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
# BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# # Move history file if present
# if HISTORY_PATH.exists():
#     shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# # Move previous training sets
# if TRAIN_DIR.exists():
#     shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# # Move any unique_sample_*.csv files
# moved_any = False
# for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
#     shutil.move(str(p), BACKUP_DIR / p.name)
#     moved_any = True

# print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())

import shutil
from pathlib import Path
from datetime import datetime

# Paths (fixed leading slashes)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")
TRAIN_DIR    = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preserve history (copy, don't move) ----
if HISTORY_PATH.exists():
    backup_history = BACKUP_DIR / HISTORY_PATH.name
    try:
        shutil.copy2(HISTORY_PATH, backup_history)
        print(f"📝 Preserved history: copied to {backup_history}")
    except Exception as e:
        print(f"⚠️ Could not copy history file: {e}")
else:
    print("ℹ️ No selection_history.json found to preserve.")

# ---- Move previous training sets to backup ----
if TRAIN_DIR.exists():
    dest = BACKUP_DIR / TRAIN_DIR.name
    try:
        shutil.move(str(TRAIN_DIR), dest)
        print(f"📦 Archived training_sets to: {dest}")
    except Exception as e:
        print(f"⚠️ Could not move training_sets: {e}")
else:
    print("ℹ️ No training_sets directory found to archive.")

# ---- Move any unique_sample_*.csv files to backup ----
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    try:
        shutil.move(str(p), BACKUP_DIR / p.name)
        print(f"📄 Archived {p.name}")
        moved_any = True
    except Exception as e:
        print(f"⚠️ Could not move {p}: {e}")

if not moved_any:
    print("ℹ️ No unique_sample_*.csv files found to archive.")

print("✅ Reset complete. Old artifacts archived. History preserved in place.")



📝 Preserved history: copied to /home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset/20251007_153240/selection_history.json
ℹ️ No training_sets directory found to archive.
📄 Archived unique_sample_20251002_043003.csv
📄 Archived unique_sample_20251007_151922.csv
✅ Reset complete. Old artifacts archived. History preserved in place.


In [3]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [4]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/miscarriage-data-updated - miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 167,
    "miscarriage": 167,
    "harassment": 166
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [5]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [6]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [7]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251007_153601.csv
Remaining after this run:
  abortion: 4077
  miscarriage: 652
  harassment: 4828


In [8]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251007_153631.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved abortion subset to: training_sets/per_class_20251007_153631/abortion_train_20251007_153631.csv
• Saved miscarriage subset to: training_sets/per_class_20251007_153631/miscarriage_train_20251007_153631.csv
• Saved harassment subset to: training_sets/per_class_20251007_153631/harassment_train_20251007_153631.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [9]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,label
0,1ku7sfe,relationships,I don't know how to get over mourning my pregn...,I'm not sure where to post this so I thought h...,2025-05-24 9:40:46,https://www.reddit.com/r/relationships/comment...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,e1104d3e92052cc00ddeaea7313942ba45217e951ade53...,abortion
1,1cdk975,relationships,I (26M) am not sure how to approach my wife (2...,So I met my wife at twenty-one years old and s...,2024-04-26 12:34:48,https://www.reddit.com/r/relationships/comment...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,5c1f54a1e633d6893eeedce8f7aedfa8fb148264dd32d6...,abortion
2,1l5ybv0,abortion,Not bleeding after taking misoprostol,"Hi everyone, just a little background I took M...",2025-06-07 23:33:12,https://www.reddit.com/r/abortion/comments/1l5...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,78ce1e45e54581069c6fed51db138e226e0fbe51e9b0d8...,abortion
3,gsyxre,abortion,What is a legitimate reason to NEED to abort a...,This may be against the rules but I'm getting ...,2020-05-29 18:58:34,https://www.reddit.com/r/abortion/comments/gsy...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,df2712ccb99b4350f56f5dada3ad3467a31739d332d550...,abortion
4,1lk8sba,TwoXChromosomes,"The Answer to The Question, ""When Was Your Las...","""Fuck the fascists. I'm not telling."" \n\nI ta...",2025-06-25 15:31:24,https://www.reddit.com/r/TwoXChromosomes/comme...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,028912d0743ddc5ce4124c1d6441dbade63343371534e2...,miscarriage
5,1lbdna3,Pregnant,My boss is overstepping,I (F24) am pregnant with my first child. I wor...,2025-06-14 17:07:37,https://www.reddit.com/r/pregnant/comments/1lb...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,4a20c595cc3c91f063ecb59158f7893ac437239622eab0...,miscarriage
6,7mmdvb,assault,Was I sexually assulted by my mother?,Im a 14 year old female with a history of self...,2017-12-28 12:43:10,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,e8a94fdb41ed793bfbbd3dc4c3517e3c73fb34c6a15803...,harassment
7,adidj1,confession,I faked my resume and now I'm in the shit........,Throwaway account for obvious reasons. Since I...,2019-01-07 14:52:59,https://www.reddit.com/r/confession/comments/a...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,3e22e5583a443a0825a81ec64063996b13d13dbbdbf1d6...,abortion
8,f2labw,metoo,I’ve been sexually assaulted and harassed seve...,I guess i just need someone to talk to and ven...,2020-02-12 4:11:25,https://www.reddit.com/r/meToo/comments/f2labw...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,91f2bf1277616af3659bff3e03e78d712d03c17878e018...,harassment
9,j0nffm,harassment,I was so blind,I’m half asleep typing this so sorry if it doe...,2020-09-27 8:07:40,https://www.reddit.com/r/SexualHarassment/comm...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,2e1ccd7f814c798ecb4637302e74d4ea58f87b2f6e670c...,harassment
